#### Goal
- To Achive Party Identification Document details from GCOB, Legacy2 For now

#### author
- Abhishek.R.Jaiswal@rabobank.com
- Prajit.tatari@rabobank.com

##### flow of logic
- Read Data from GCOB and Legacy2 to further transform if needed

##### Version & Changes
|     Developer |Date	   | PBI/Bug No |	Changes done |
|----------|----------|----------|----------|
| Abhishek Jaiswal	 |3-July-2025 |12950919 |First release
| Prajit	 |11-Aug-2025 |13353847 |Bug Fixes
| Abhishek Jaiswal	 |16-Sep-2025 |13584702 |Added CountryOfIssueIsoCode
| Mahalakshmi V | 20-Nov-2025 | 14165735 | Made notebook compatible for Historical Load
| Hari | 20-May-2026 | 16070036 | Read RanzV4  Data for Radar V3



#####Read Files from GDP

In [0]:
import os
app_reg_app_id = os.environ['APP_REG_APP_ID']
ReadStorage = os.environ['GDP_STORAGE_NAME']
TenantId = os.environ['TENANT_ID']
GIC_ReadStorage = os.environ['GDP_SA_STORAGE_NAME']
environment=os.environ['ENV']
radar_datamodel_version_number=3
service_credential = dbutils.secrets.get(scope="connectedsecrets", key = f"appreg-{app_reg_app_id}")
AU_GDP_Defined_Storage_Account=os.environ['AU_GDP_Defined_Storage_Account']
SARADAR = "saradar" + environment

In [0]:
from RadarUtils import *

In [0]:
dbutils.widgets.text("Load_Date", "")
dbutils.widgets.text("RunType", "daily")  # default is daily

Load_Date = dbutils.widgets.get("Load_Date")
RunType = dbutils.widgets.get("RunType").lower()
if Load_Date:
    RunType="historical"
    
if not Load_Date:
    Load_Date = datetime.today().strftime('%Y%m%d')

print(f"Running for Load Date: {Load_Date}")

In [0]:
party_dataobject='Party_Identification'

In [0]:
authenticate_storage_account(ReadStorage)
authenticate_storage_account(GIC_ReadStorage)
authenticate_storage_account(SARADAR)
authenticate_storage_account(AU_GDP_Defined_Storage_Account)

In [0]:
# Derive the date for which data has to be processed from GDP
load_dts = f"EDL_LOAD_DTS={Load_Date}*"
print (load_dts)

In [0]:
# print list of strings for loading spark dfs from GDP
load_df =[
'party_RelatedPartyIdentificationDocument'
,'party_case_client_details'
,'party_allcountries'
]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='GCOB', Dataobject=Object, path_prefix='CaseService',Load_Date=Load_Date)

In [0]:
# List of datasets from GDP
load_df = [
'Legacy2_ClientIdentification',
'Legacy2_case_client_details'
]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='Legacy2', Dataobject=Object,path_prefix='Legacy2',Load_Date=Load_Date)

In [0]:
# List of dataobjects from GDP
RANZ_Dataobjects_List = [
"c_b_party_identifier_xref",
"c_b_party_xref"
]

# Create TempView for each loading table
# for Object in RANZ_Dataobjects_List:
load_ranz_v4_rdm_tables(RANZ_Dataobjects_List,Load_Date)

In [0]:
df_Party_SystemIdentifier = spark.read.parquet(
    f"abfss://radardatamodel@{SARADAR}.dfs.core.windows.net/Party_SystemIdentifier/3/data/{load_dts}/*.parquet"
)

#####GCDS - Other Sources Data Prepration

In [0]:
%sql
Create or replace temporary view Gcob_IdentificationDocument As
with Indentification as 
(
select distinct Ident.*
,case when c.CaseStatusName is null then 'RelatedParty' else c.CaseStatusName end as StatusName
FROM party_RelatedPartyIdentificationDocument AS Ident 
inner join (select distinct UniquePartyId, max(partyid) as Partyid from party_RelatedPartyIdentificationDocument
where expired is null
group by UniquePartyId) identification
on Ident.UniquePartyId = identification.UniquePartyId and Ident.Partyid = identification.Partyid
left join party_case_client_details c on Ident.UniquePartyId = c.UniqueGcobId
WHERE Ident.IdType IS NOT NULL and identification.UniquePartyId is not null
)
SELECT DISTINCT
CONCAT('GCOB_', Ident.UniquePartyId) AS LocalSystemIdentifier,
'GCOB' as Application,
Ident.IdType,
Ident.DescriptionOfOtherIdentificationType as IdTypeOtherDescription,
Ident.IdNumber,
Ident.PlaceOfIssue,
Ident.CountryOfIssue,
to_date(Ident.IssueDate, 'dd-MM-yyyy') as IssueDate,
to_date(Ident.ExpiryDate, 'dd-MM-yyyy') as ExpiryDate
FROM Indentification AS Ident 
WHERE StatusName <> 'Cancelled'

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW NonSFDCID_Legacy2_Client AS
select * from Legacy2_case_client_details where isclient = 1 and GcobCaseId is null and Value is null and GcobId like 'RA:%';

CREATE OR REPLACE TEMPORARY VIEW Legacy2_client AS
select * 
,CASE
  WHEN IsClient = 'true' and  ClientTypeId = 1 THEN concat('LE_', GcobId)
  WHEN IsClient = 'true' and ClientTypeId in (2,3) THEN concat('NP_NPPC_', GcobId)
  WHEN IsClient = 'false' and ClientTypeId = 1 THEN concat('RLE_', GcobId)
  WHEN IsClient = 'false' and ClientTypeId in (2,3) THEN concat('RNP_', GcobId)
END as Legacy2_Identifier
from Legacy2_case_client_details
where ClientId not in (select ClientId from NonSFDCID_Legacy2_Client)
;

CREATE OR REPLACE TEMPORARY VIEW Legacy2_IdentificationDocument AS
select distinct
concat('LEGACY2_',c.Legacy2_Identifier) as LocalSystemIdentifier,
'Legacy2' as Application,
Ident.IdType,
null as IdTypeOtherDescription,
Ident.ContactIdentificationNumber as IdNumber,
Ident.ContactPlaceOfIdIssue as PlaceOfIssue,
null as CountryOfIssue,
-- Ident.ContactDateOfIdIssue as IssueDate,
-- Ident.ContactIdExpiryDate as ExpiryDate
to_date(Ident.ContactDateOfIdIssue, 'dd-MM-yyyy') as IssueDate,
to_date(Ident.ContactIdExpiryDate, 'dd-MM-yyyy') as ExpiryDate
from Legacy2_ClientIdentification Ident
inner join Legacy2_client c on Ident.id = c.ClientId

In [0]:
%sql

CREATE OR REPLACE TEMPORARY VIEW RANZ_IdentificationDocument AS
select 
concat('RANZ_',Ranz_Party.SRC_PARTY_ID) as LocalSystemIdentifier
,'RANZ' as Application
,idnt.ID_TYPE_CD as IDType
,null as IdTypeOtherDescription
,idnt.ID_VALUE as IdNumber
,idnt.PLACE_OF_ISSUE as PlaceOfIssue
,idnt.ID_CNTRY_CD as CountryOfIssueISOCode
,idnt.ISSUE_DT as IssueDate
,idnt.ID_EXPIRY_DT as ExpiryDate
from c_b_party Ranz_Party
left join  c_b_party_identifier idnt on idnt.FK_PARTY_ID=Ranz_Party.ROWID_XREF




In [0]:
%sql
Create or replace temporary view Party_Identifier As
select * From Gcob_IdentificationDocument
union
select * From Legacy2_IdentificationDocument
union
select * from RANZ_IdentificationDocument

In [0]:
df_party_Identifier=spark.table('Party_Identifier')

In [0]:
df_party_Identifier = add_party_identifier(df_party_Identifier, df_Party_SystemIdentifier)

In [0]:
df_party_Identifier.createOrReplaceTempView('Identifier')

In [0]:
%sql
Create or replace temporary view Final_Party_Identifier As
with non_legacy2_duplicates as (
select distinct * from Identifier 
where PartyIdentifier is not null
qualify rank() over(partition by PartyIdentifier order by Application) = 1
)
,all_countries as (
  select distinct Name as CountryName, ISOCode from party_allcountries
order by CountryName
)
select distinct
d.*
,c.ISOCode as CountryOfIssueISOCode
from non_legacy2_duplicates d
left join all_countries c on d.CountryOfIssue = c.CountryName
qualify row_number() over(partition by PartyIdentifier,IDNumber order by IDNumber, issuedate desc) = 1

In [0]:
df_Final_Identifier=spark.table('Final_Party_Identifier')

In [0]:

if RunType == "historical":
    save_to_saradar_storage_account(df_Final_Identifier, party_dataobject, radar_datamodel_version_number, environment,Load_Date)
else:
    save_to_saradar_storage_account(df_Final_Identifier, party_dataobject, radar_datamodel_version_number, environment)